#BERTopic on UDC Sample Dataset
In this notebook I have implemented Topic Modeling on Urdu News Data using a transformer based topic modeling technique BERTopic.

## Mounting Google Drive
If the dataset is on Google Drive then you have to mount over google drive with collaboratory.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive




#Installing required dependencies
**One thing to remember is that after installing libraries you have to restart the run time again so that other dependencies are not affected by it.**

In [ ]:
!pip install bertopic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.1/154.1 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 19.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.9/90.9 kB 10.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.8/132.8 kB 15.0 MB/s eta 0:00:00
  Using cached Cython-0.29.37-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.manylinux_2_24_x86_64.whl (1.9 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 7.1 MB/s eta 0:00:00
  Created wheel for hdbscan: filename=hdbscan-0.8.33-cp310-cp310-linux_x86_64.whl size=3039292 sha256=e6303bb0726573b8a4f911ebeb863784c8e3d33a5dcd03c9fc3d5ef9c8ec59be
  Stored in directory: /root/.cache/pip/wheels/75/0b/3b/dc4f60b7cc455efaefb62883a7483e76f09d06ca81cf87d610
  Created wheel for umap-

In [ ]:
!pip install -U sentence-transformers

In [ ]:
!pip install --upgrade tensorflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.2/475.2 MB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 40.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.0/442.0 kB 9.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 45.8 MB/s eta 0:00:00
  Attempting uninstall: tensorflow-estimator
    Found existing installation: tensorflow-estimator 2.12.0
    Uninstalling tensorflow-estimator-2.12.0:
      Successfully uninstalled tensorflow-estimator-2.12.0
  Attempting uninstall: keras
    Found existing installation: keras 2.12.0
    Uninstalling keras-2.12.0:
      Successfully uninstalled keras-2.12.0
  Attempting uninstall: google-auth-oauthlib
    Found existing installation: google-auth-oauthlib 0.4.6
    Uninstalling google-auth-oauthlib-0.4.6:
      Successfully uninstalled google-auth-oauthlib-0.4.6
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.12.0
    Uninstalling ten


# Importing required dependencies
We will import numpy, pandas and re, bertopic, gensim library for now. other libraries will be imported in the notebook later.

Pandas will be used to create a Dataframe and handle the csv file. Numpy will be used for the faster computation of arrays to save time. re library will be used for the cleaning of data. gensim library is used to get coherence score. bertopic is used to train bertopic on UDC dataset with using pretrained language models

In [ ]:
import pandas as pd
import numpy as np
import re
from bertopic import BERTopic
from gensim.models.coherencemodel import CoherenceModel
import gensim.corpora as corpora
#optional
import logging
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.ERROR)

import warnings
warnings.filterwarnings("ignore",category=DeprecationWarning)

##DataFrame





In [ ]:
#load UNTM Dataset
df = pd.read_csv('/content/drive/MyDrive/sampled_UDC.csv', encoding='utf-8')

In [ ]:
print(len(df))
df.head()

101


,Document,Category,Clean_Data
0,بی آئی ایس پی نے بجٹ میں 100 فیصد اضافہ مانگ...,Business,بی آئی ایس پی نے بجٹ میں فیصد اضافہ مانگ لی...
1,پی ایس او کے مختلف اداروں پر واجبات 335 ارب س...,Business,پی ایس او کے مختلف اداروں پر واجبات ارب سے ت...
2,کراچی: تھائی ایئر ویز انٹرنیشنل کے کنٹری مینی...,Business,کراچی تھائی ایئر ویز انٹرنیشنل کے کنٹری مینیج...
3,اسلام آباد: وفاقی حکومت کا قرضہ صرف 8ماہ میں...,Business,اسلام آباد وفاقی حکومت کا قرضہ صرف ماہ میں ...
4,اسلام آباد: وفاقی حکومت کی جانب سے اعلان کرد...,Business,اسلام آباد وفاقی حکومت کی جانب سے اعلان کردہ...


## Preprocessing of Data
Data was cleaned that saved into column "Clean_Data"  We further remove some unnecessary words.
Stopwords are common words that are often filtered out during text processing in natural language processing (NLP) tasks. These words are considered to have little or no value in conveying the actual meaning of the text. We take list of 401 stopwords for topic modelling.

In [ ]:
df['Clean_Data'] = df['Clean_Data'].str.replace('شیئرٹویٹشیئرای میلتبصرے مزید شیئر', '')

In [ ]:
df['Clean_Data'] = df['Clean_Data'].str.replace('کیلیے', '')

In [ ]:
def remove_nonbreaking_space(text):
    return re.sub(r'\xa0', ' ', text)

df['Clean_Data'] = df['Clean_Data'].apply(remove_nonbreaking_space)

In [ ]:
documents = df['Clean_Data'].values.tolist()
print((documents[1]))
doc1=documents[1]

 پی ایس او کے مختلف اداروں پر واجبات  ارب سے تجاوز پی ایس او کے مختلف اداروں پر واجبات  ارب سے تجاوزشیئرٹویٹ علیم ملک  پير  اپريل    ارب  کروڑ کے ساتھ پاورسیکٹرسب سے بڑا نادہندہپی آئی اے کے ذمے  ارب۔ فوٹو فائل   ارب  کروڑ کے ساتھ پاورسیکٹرسب سے بڑا نادہندہپی آئی اے کے ذمے  ارب۔ فوٹو فائل   اسلام آباد پاکستان اسٹیٹ آئل  کا مالی بحران سنگین ہوگیا ہے اورپی ایس او کے مختلف اداروں کے ذمے واجبات  ارب کروڑ روپے کی ریکارڈ سطح پر پہنچ گئے ہیں۔  مختلف اداروں کی جانب سے پی ایس او کو باقاعدگی سے ادائیگی نہ کرنے کی وجہ سے پی ایس او مالیاتی بحران کا شکار ہوگیاہے اور عدم ادائیگیوں کی جانب سے پی ایس او مشکلات کا شکار ہوگیا ہے۔ دستاویز کے مطابق مختلف اداروں کی جانب سے پاکستان اسٹیٹ آئل کو  ارب سے زائد کی رقم واجب الادا ہے۔  پاور سیکٹر پی ایس او کا سب سے بڑا نا دہندہ ہے اور پاور سیکٹر نے پی ایس او کو  ارب  کرروڑ روپے کی ادائیگیاں کرنی ہیں۔ سرکاری دستاویز کے مطابق بجلی کی پیداواری کمپنیوں  نے پی ایس او کو ایک سو ترپن ارب روپے حبکو نے تراسی ارب چالیس کروڑ روپے اور کو ٹ ادو پاور کمپنی کیپکو نے پی ایس 

In [ ]:
import nltk
stopwords=pd.read_csv('/content/drive/MyDrive/stopwords.txt',names=['List'])
# stopwords['List']

stopwords_ur=[]
for li in stopwords['List']:
  stopwords_ur.append(li)
print(len(stopwords_ur))
print(stopwords_ur)

401
['کی', 'ہیں', 'ہے', 'رہا', 'رہی', 'رہے', 'تھا', 'تھے', 'تھی', 'تھى', 'میں', 'کہ', 'کے', 'ہوتے', 'کہہ', 'بنانا', 'پھر', 'لکین', 'ہوتی', 'لیتی', 'ایسی', 'ائے', 'ہوئے', 'ہوۓ ', 'مگر', 'چاہے', 'کیے', 'تاکہ', 'تم', 'آکر', 'لگا', 'ہوگیئں', 'ليے', 'رہتی', 'ہوگئی', 'انھوں', 'چاہتے', 'پاگیئں', 'آنا', 'کروا', 'رکھ', 'آتی', 'یہاں', 'جیسا', 'چکے', 'کرئے', 'دیے', 'چکا', 'ملتا', 'کبھی', 'ایسا', 'کرسکیں', 'ہوسکے', 'سکیں', 'لہذا', 'چاہئے', 'وہیں', 'اٹھایا', 'جہنوں', 'ہمارے', 'لیے', 'آرہے', 'کرگیئں', 'بنانے', 'سکتا', 'وغیرہ', 'دے', 'ہوۓ', 'رہنے', 'ہوۓ', 'کئے', 'لگے', 'لگایا', 'لائے', 'کہے', 'کرے', 'رہئں', 'آگئے', 'کئی', 'کم', 'ملی', 'جنہیں', 'ہوئیں', 'تھیں', 'ابھی', 'پاگئیں', 'آئے', 'کرا', 'دیا', 'جہاں', 'آگئیں', 'کرتی', 'رہیں', 'کرتیں', 'دیتی', 'ہوگئے', 'ديتے', 'انہں', 'ایسے', 'رکھتے', 'رہتے', 'رکھی', 'کیں', 'كیا', 'وه', 'جنہيں', 'کروائی', 'دینا', 'جسے', 'جاتی', 'اسکی', 'ملے', 'کرگئى', 'جن', 'آپکو', 'جنہوں', 'دیئے', 'والی', 'جبکہ', 'دیگر', 'آپکا', 'رکھنے', 'انہىں', 'کيے', 'یعنی', 'كيے', 'بننے', 'ج

# BERTopic Training


In [ ]:
#create custom embedding
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/distiluse-base-multilingual-cased-v2')
embeddings = model.encode(documents, show_progress_bar=True)
print(embeddings)

modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.69k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/610 [00:00<?, ?B/s]

...

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2_Dense/config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

[[ 0.00916042  0.06687659  0.00793992 ...  0.03491772 -0.02360609
   0.01003345]
 [ 0.04833498  0.00919292  0.00568508 ...  0.02414999 -0.02264939
  -0.00885205]
 [ 0.01044563  0.04916572 -0.02431903 ...  0.02065906 -0.02522849
   0.03500259]
 ...
 [ 0.0157033   0.07207946 -0.05418859 ...  0.00750634 -0.02253509
  -0.01156019]
 [ 0.00571777  0.00127601  0.03927448 ... -0.04930213  0.01686148
  -0.00087841]
 [-0.00262932  0.04106253  0.0123721  ...  0.02529592  0.01291248
   0.02558282]]


In [ ]:
#pass vectorizer_model to bertopic for stopwords removal
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model = CountVectorizer(stop_words= stopwords_ur)


In [ ]:
#ClassTfidf for top words
from bertopic.vectorizers import ClassTfidfTransformer

ctfidf_model = ClassTfidfTransformer(bm25_weighting=True, reduce_frequent_words=True)

In [ ]:
#UMAP for dimention reduction
from umap import UMAP
dim_model = UMAP(n_neighbors=10, n_components=5, random_state=42)

In [ ]:
# # #KMeans used for clustering
from sklearn.cluster import KMeans

cluster_model = KMeans(n_clusters=5, random_state=42)

In [ ]:
np.random.seed(42)

In [ ]:
topic_model = BERTopic(language="urdu", low_memory=True ,calculate_probabilities=True, vectorizer_model=vectorizer_model, top_n_words=10, umap_model=dim_model,ctfidf_model=ctfidf_model, hdbscan_model=cluster_model)

In [ ]:
#Fit documents in bertopic
topics, probs = topic_model.fit_transform(documents,embeddings)

In [ ]:
print(probs)

None


In [ ]:
#topics that assign to each document
print(topics)

[2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 2, 2, 0, 4, 0, 2, 4, 2, 2, 2, 2, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 3, 3, 3, 3, 3, 0, 0, 3, 0, 3, 3, 0, 3, 0, 3, 3, 3, 3, 3, 1, 4, 4, 4, 1, 4, 3, 4, 4, 0, 4, 4, 4, 0, 4, 4, 4, 4, 4, 4, 0, 3, 0, 0, 3, 3, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0, 0]


In [ ]:
topic_model.get_topic_freq()

,Topic,Count
1,0,28
3,1,20
0,2,18
4,3,18
2,4,17


In [ ]:
document_topics = topic_model.get_topics()

In [ ]:
#topics with score
print(document_topics)

{0: [('استعمال', 0.3062610610546161), ('رنز', 0.2979714722489764), ('گھر', 0.2971189246116796), ('دور', 0.2965064757699254), ('دنیا', 0.28433004325102396), ('قدیم', 0.28191209596494216), ('قبل', 0.27461843013320864), ('پہلے', 0.2724209446433291), ('دریافت', 0.26450022940360346), ('نسو', 0.26294640892763005)], 1: [('فلم', 0.5126266432704637), ('بالی', 0.49380481602308285), ('ووڈ', 0.4780604250806612), ('اداکارہ', 0.450719691188612), ('خان', 0.44361403866713917), ('میڈل', 0.4393999617717059), ('ساتھ', 0.4368771041452682), ('صفہ', 0.3908982155994856), ('ماہرہ', 0.3814331344882132), ('میٹرز', 0.3814331344882132)], 2: [('ارب', 0.5862166159439897), ('ٹیکس', 0.5561308496210363), ('بینک', 0.5258126937644019), ('روپے', 0.5208649385657871), ('لاکھ', 0.5116976890154278), ('کروڑ', 0.500101867664878), ('مالی', 0.49376204584816), ('او', 0.48940136384641403), ('پی', 0.4628873613765677), ('ایس', 0.4509818699341509)], 3: [('کینسر', 0.47368497386964964), ('منہ', 0.45296663480411903), ('ڈاکٹر', 0.3906246

In [ ]:
#shows which documents assign to specific topic
topic_model.get_representative_docs(topic=None)

{0: ['  انہیں کھانے سے ذائقے کے ساتھ ساتھ خوشگواریت کا بھی احساس ہوتا ہے۔ کھانوں میں ان مسالوں کی موجودگی کا خیال رکھنا بہت ضروری ہے کیوں کہ یہ مسالے خوش ذائقہ ہونے کے ساتھ ساتھ مفید بھی ہیں۔ انہیں اچھی طرح صاف کرکے ڈبوں میں محفوظ کر لیا جائے تاکہ بہ وقت ضرورت کام آسکیں۔  یہاں ہم چیدہ چیدہ مسالوں کے فوائد اور ان کے مفید استعمال کا ذکر کرتے ہیں تاکہ آپ ان مسالوں کی اہمیت سے آگاہ ہوسکیں اور اپنے کھانوں کو مزے دار بنانے میں ان کی مدد لے سکیں۔ کلونجی کلونجی کے چند دانے کھانے میں ضرور استعمال کرنے چاہییں کلونجی میں بہت سی بیماریوں کا علاج موجود ہے۔ معدے اور پیٹ کے امراض کے علاوہ آنتوں کے درد دماغی کم زوری اور فالج سے بچاؤ کا حل بھی کلونجی میں موجود ہے۔ اچار میں اس کا استعمال کیا جاتا ہے بلکہ اچار اس کے بغیر مکمل نہیں ہوتا۔ اس میں کاربوہائیڈریٹ فاسفورس اور فولاد کے مرکبات پائے جاتے ہیں۔  ایک تجزیاتی رپورٹ کے مطابق کلونجی میں ایک پیلے رنگ کا مادہ کیروٹینبھی پایا جاتا ہے جو جگر میں پہنچ کر وٹامن اے میں تبدیل ہوجاتا ہے۔ کلونجی جسم کی قوت مدافعت کو بڑھاتی ہے۔ ایک کپ گرم پانی میں نصف چائے کا 

In [ ]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,28,0_استعمال_رنز_گھر_دور,"[استعمال, رنز, گھر, دور, دنیا, قدیم, قبل, پہلے...",[ انہیں کھانے سے ذائقے کے ساتھ ساتھ خوشگواریت...
1,1,20,1_فلم_بالی_ووڈ_اداکارہ,"[فلم, بالی, ووڈ, اداکارہ, خان, میڈل, ساتھ, صفہ...",[ بالی ووڈ کے نامور اداکار رنویر سنگھ فلموں م...
2,2,18,2_ارب_ٹیکس_بینک_روپے,"[ارب, ٹیکس, بینک, روپے, لاکھ, کروڑ, مالی, او, ...",[ ارشاد انصاری اتوار اپريل بجٹ میں کئی اقد...
3,3,18,3_کینسر_منہ_ڈاکٹر_خون,"[کینسر, منہ, ڈاکٹر, خون, پاکستان, مریضوں, فالج...",[ فالج کا حملہ اس وقت ہوتا ہے جب دماغ کو خون ...
4,4,17,4_گیمز_سندھ_محمد_پاکستان,"[گیمز, سندھ, محمد, پاکستان, برانز, جی, سلور, گ...",[ پاکستان کو گذشتہ برس سری لنکا سے آخری ٹیسٹ...


In [ ]:
topic_model.get_document_info(documents)

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Representative_document
0,بی آئی ایس پی نے بجٹ میں فیصد اضافہ مانگ لی...,2,2_ارب_ٹیکس_بینک_روپے,"[ارب, ٹیکس, بینک, روپے, لاکھ, کروڑ, مالی, او, ...",[ ارشاد انصاری اتوار اپريل بجٹ میں کئی اقد...,ارب - ٹیکس - بینک - روپے - لاکھ - کروڑ - مالی ...,False
1,پی ایس او کے مختلف اداروں پر واجبات ارب سے ت...,2,2_ارب_ٹیکس_بینک_روپے,"[ارب, ٹیکس, بینک, روپے, لاکھ, کروڑ, مالی, او, ...",[ ارشاد انصاری اتوار اپريل بجٹ میں کئی اقد...,ارب - ٹیکس - بینک - روپے - لاکھ - کروڑ - مالی ...,True
2,کراچی تھائی ایئر ویز انٹرنیشنل کے کنٹری مینیج...,2,2_ارب_ٹیکس_بینک_روپے,"[ارب, ٹیکس, بینک, روپے, لاکھ, کروڑ, مالی, او, ...",[ ارشاد انصاری اتوار اپريل بجٹ میں کئی اقد...,ارب - ٹیکس - بینک - روپے - لاکھ - کروڑ - مالی ...,False
3,اسلام آباد وفاقی حکومت کا قرضہ صرف ماہ میں ...,2,2_ارب_ٹیکس_بینک_روپے,"[ارب, ٹیکس, بینک, روپے, لاکھ, کروڑ, مالی, او, ...",[ ارشاد انصاری اتوار اپريل بجٹ میں کئی اقد...,ارب - ٹیکس - بینک - روپے - لاکھ - کروڑ - مالی ...,False
4,اسلام آباد وفاقی حکومت کی جانب سے اعلان کردہ...,2,2_ارب_ٹیکس_بینک_روپے,"[ارب, ٹیکس, بینک, روپے, لاکھ, کروڑ, مالی, او, ...",[ ارشاد انصاری اتوار اپريل بجٹ میں کئی اقد...,ارب - ٹیکس - بینک - روپے - لاکھ - کروڑ - مالی ...,False
...,...,...,...,...,...,...,...
96,ایسیکس کی رہائشی ٹورز رینلڈز نے بتایا کہ وہ ...,0,0_استعمال_رنز_گھر_دور,"[استعمال, رنز, گھر, دور, دنیا, قدیم, قبل, پہلے...",[ انہیں کھانے سے ذائقے کے ساتھ ساتھ خوشگواریت...,استعمال - رنز - گھر - دور - دنیا - قدیم - قبل ...,False
97,زیر نظر مضمون میں یہ کوشش کی گئی ہے کہ ان چی...,0,0_استعمال_رنز_گھر_دور,"[استعمال, رنز, گھر, دور, دنیا, قدیم, قبل, پہلے...",[ انہیں کھانے سے ذائقے کے ساتھ ساتھ خوشگواریت...,استعمال - رنز - گھر - دور - دنیا - قدیم - قبل ...,True
98,غیر ملکی خبر رساں ادارے کے مطابق مصر کے شہر ...,0,0_استعمال_رنز_گھر_دور,"[استعمال, رنز, گھر, دور, دنیا, قدیم, قبل, پہلے...",[ انہیں کھانے سے ذائقے کے ساتھ ساتھ خوشگواریت...,استعمال - رنز - گھر - دور - دنیا - قدیم - قبل ...,False
99,مقامی میڈیا کے مطابق ری لیو ء میں ترکی پہنچا...,0,0_استعمال_رنز_گھر_دور,"[استعمال, رنز, گھر, دور, دنیا, قدیم, قبل, پہلے...",[ انہیں کھانے سے ذائقے کے ساتھ ساتھ خوشگواریت...,استعمال - رنز - گھر - دور - دنیا - قدیم - قبل ...,False


In [ ]:
topic_distr, _ = topic_model.approximate_distribution(documents, window=3, min_similarity=0.01)

In [ ]:
print(topic_distr)

[[0.11043388 0.08970342 0.53260223 0.13086717 0.13639329]
 [0.12836009 0.07468817 0.55017893 0.11840401 0.1283688 ]
 [0.12226354 0.15802625 0.45874011 0.10914482 0.15182527]
 [0.11350355 0.09710731 0.52281194 0.13312146 0.13345574]
 [0.10493059 0.05386726 0.60581182 0.13426541 0.10112492]
 [0.13159524 0.11004025 0.49889632 0.09800484 0.16146335]
 [0.09199096 0.06750497 0.59979011 0.08452141 0.15619256]
 [0.12985309 0.05334421 0.6193659  0.06490294 0.13253386]
 [0.13596989 0.07979552 0.51870151 0.14502809 0.12050499]
 [0.40403018 0.10713312 0.17364689 0.18921167 0.12597814]
 [0.10066036 0.06244056 0.59158469 0.1129148  0.13239958]
 [0.12487921 0.08441333 0.54817729 0.12823979 0.11429038]
 [0.31535548 0.0906678  0.27743639 0.15465104 0.16188929]
 [0.13400239 0.08995014 0.23627803 0.12649631 0.41327314]
 [0.35268588 0.08981247 0.29507203 0.11102903 0.15140059]
 [0.12137293 0.09856914 0.48123156 0.15257746 0.1462489 ]
 [0.10989163 0.09765875 0.19682453 0.16591337 0.42971173]
 [0.11799234 0

In [ ]:
topic_model.visualize_distribution(topic_distr[0], width=600,height=600,  title="Topic Probability Distribution (UDC)")

# Evaluation
we used three evaluation metrics to compare the results.

1. The coherence score is used to capture the degree of similarity between the words within each topic, with higher scores indicating more coherent topics. We used two coherence metrics NPMI and Cv Score.
2. IRBO measures are used to assess how different and distinct the topics are in a topic model.


### Coherence Score
To evaluate the model topics coherence we use [Gensim](https://radimrehurek.com/gensim/models/coherencemodel.html) library

In [ ]:
texts = [[word for word in str(document).split() if word not in stopwords_ur] for document in documents]
id2word = corpora.Dictionary(texts)
corpus = [id2word.doc2bow(text) for text in texts]

In [ ]:
topics_bert=[]
for i in topic_model.get_topics():
  row=[]
  topic= topic_model.get_topic(i)
  for word in topic:
     row.append(word[0])
  topics_bert.append(row)

In [ ]:
print(topics_bert)

[['کینسر', 'استعمال', 'جسم', 'علاج', 'افراد', 'پانی', 'ماہرین', 'صحت', 'خواتین', 'تحقیق'], ['ٹیم', 'کرکٹ', 'محمد', 'میڈل', 'انگلینڈ', 'گولڈ', 'ٹیسٹ', 'میچ', 'کھلاڑیوں', 'گیمز'], ['روپے', 'ارب', 'فیصد', 'ٹیکس', 'حکومت', 'پاکستان', 'بجٹ', 'ڈالر', 'کروڑ', 'مالی'], ['فلم', 'خان', 'ووڈ', 'بالی', 'اداکارہ', 'سلمان', 'بھارتی', 'اداکار', 'میڈیا', 'انڈسٹری'], ['ظفر', 'میشا', 'شفیع', 'علی', 'جنسی', 'ہراسانی', 'الزام', 'گلوکارہ', 'مومنہ', 'ہراساں']]


In [ ]:
# compute Coherence Score

cm = CoherenceModel(topics=topics_bert, texts=texts, dictionary=id2word, coherence='c_npmi')
coherence = round(cm.get_coherence(),2)
print('\nCoherence Score: ', coherence)


Coherence Score:  -0.08


**Diversity Score**

In [ ]:
import itertools
from rbo import rbo
import numpy as np

class InvertedRBO:
    def __init__(self):
        pass

    def irbo(self, topics, topk=10, weight=0.9):
        """
        Calculate inverted Rank Biased Overlap (RBO) as a measure of topic diversity from a list of lists of words.

        :param topics: A list of lists of words representing different topics.
        :param topk: The number of top words on which RBO will be computed.
        :param weight: Weight of each agreement at depth d: p**(d-1). When set to 1.0, there is no weight,
                       and the RBO returns to average overlap.
        :return: The inverted RBO topic diversity score.
        """
        if topk <= 0:
            raise ValueError("topk must be a positive integer.")

        num_topics = len(topics)
        if num_topics == 0:
            raise ValueError("topics list cannot be empty.")

        if topk > len(topics[0]):
            raise Exception('Words in topics are less than topk')

        collect = []
        for list1, list2 in itertools.combinations(topics, 2):
            rbo_val = rbo(list1[:topk], list2[:topk], p=weight)[2]
            collect.append(rbo_val)

        Irbo_score = 1 - np.mean(collect)
        return Irbo_score

In [ ]:
inverted_rbo_calculator = InvertedRBO()
IRBO= round(inverted_rbo_calculator.irbo(topics_bert, topk=10, weight=0.9),2)
print("Inverted RBO Score:", IRBO)

Inverted RBO Score: 0.99


# Visualize Topics

In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_barchart(n_words=10, width=220, height=270, title= "Topic Word Scores (UDC)")